# Generate response from Language model 

In [1]:
import ollama # Used to load model .
from textwrap import dedent # Used for spacing problems in prompt .
from tabulate import tabulate # Used for creating a table for displaying models .
import subprocess # Used to start ollama server .
import time # For waiting .
import requests # Used to access ollama server .
from PIL import Image # For image datatype .
import base64 # For conversion of PIL image to base64 .
import io # For I/O operation of image .
from typing import Optional , List,Dict # For datatype validation .
import tiktoken
from dotenv import load_dotenv

### Ollama Setup Notes

* Start the Ollama server **before** running the code.
  Running the code multiple times without a running server can create multiple Ollama instances, which may waste RAM and cause the model to fail.

  ```bash
  ollama serve
  ```

* Download the required model locally before running the code.

  ```bash
  ollama pull <model-name>
  ```


In [2]:
load_dotenv()

True

In [3]:
# Removing unwanted warnings from sentence transformers library .(Optional)
import warnings
from transformers import logging

logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*position_ids.*UNEXPECTED.*")

In [4]:
from sentence_transformers import SentenceTransformer # Used for loading model .
import numpy as np # Used to store embedding .
import torch # Used for device selection and model execution .
from typing import List # Used for return type .
from langchain_core.documents import Document # Used for storing documents .

In [5]:
# Used for loading model and embedding text .
class TextEmbeddingModel:
    def __init__(self, model_name: str = "BAAI/bge-large-en-v1.5"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # Device check.

        try:
            self.model = SentenceTransformer(model_name_or_path=model_name, device=self.device)  # Loading model.
        except Exception as e:
            raise RuntimeError(f"Failed to load SentenceTransformer model: {e}")

        self.query_prefix = "Represent this sentence for searching relevant passages: "

        if self.device == "cuda":
            print(f"BGE running on {torch.cuda.get_device_name(0)}.")
        else:
            print("BGE running on CPU.")
        print(f"Embedding dimension of {model_name} is {self.model.get_sentence_embedding_dimension()}")

    @torch.no_grad()
    def embed_query(self, query: str) -> np.ndarray:
        if not isinstance(query, str):  # Checking if query is valid.
            raise TypeError("Query must be a string.")

        if not query.strip():  # Checking if the prompt is not empty.
            raise ValueError("Given prompt is not valid.")

        query = self.query_prefix + query  # Adding a prefix to prompt since BGE is instruction-based (only for query, not documents).

        query_embedding = self.model.encode(  # Embedding query.
            sentences=query,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        return query_embedding  # Returning embedded values.

In [6]:
from sentence_transformers import util
import re

In [7]:
# Used to initialize a language model and generate responses .
class LocalLLM:
    def __init__(self, model_name: str = "gemma3:4b", text_embedder: TextEmbeddingModel = None):
        # Default model; can change.
        self.model_name = model_name
        self.process = self.ollama_server(process="start")
        self.text_embedder = text_embedder or TextEmbeddingModel()
        if not self.is_model_available(self.model_name):
            # Checking if model is available or valid.
            available = [m['model_name'] for m in self.available_models()]
            raise ValueError(
                f"Model '{model_name}' not available. "
                f"Available models: {available}"
            )
        self.tokenizer = tiktoken.get_encoding("cl100k_base")

    # The following function is used to start or stop ollama server.
    def ollama_server(self, process: str):
        if process == "start":
            try:
                requests.get("http://localhost:11434/api/tags", timeout=1)  # Checking if ollama server is already started.
                print("Ollama already running")
                return "external"
            except:
                pass
            ollama_process = subprocess.Popen(  # Starting ollama server if not started.
                ["ollama", "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                shell=False
            )
            for _ in range(10):  # Checking if server started.
                try:
                    requests.get("http://localhost:11434/api/tags", timeout=1)
                    print("Ollama server started")
                    return ollama_process
                except:
                    time.sleep(1)
            raise RuntimeError("Ollama failed to start")  # If server did not start after many tries, raise error.

        elif process == "stop":  # Stopping ollama server.
            if isinstance(self.process, subprocess.Popen):  # Checking if server was started here.
                self.process.terminate()
                self.process.wait()
                print("Ollama server successfully stopped.")
            else:
                print("Ollama was not started by this process")  # If started externally, notify.
            return None
        else:
            raise ValueError("Input can be either 'start' or 'stop'")  # Input validation.

    # The following function is used to check available models in the local device.
    def available_models(self):
        response = ollama.list()  # Getting available models.
        models_available = response.get('models', [])  # Safer access
        models = []
        for m in models_available:  # Getting required information from model.
            models.append({
                "model_name": m.get('model'),
                "parameters": m.get('details', {}).get('parameter_size')
            })
        return models if models else []

    # The following function is to check if a specific model is available in local device.
    def is_model_available(self, model_name):
        models = self.available_models()
        return any(m['model_name'] == model_name for m in models)

    # The following function is used to build prompt using user query and retrieved documents.
    # Enhanced prompt with chain-of-thought reasoning for better quality answers.
    def build_prompt(self, query: str, context: str):
        return f"""You are a rigorous scientific analyst specialized in PDF documents with images and captions. Your task is to answer the question using ONLY the provided context, images, and captions. You must remain strictly evidence-based and avoid speculation. You are not allowed to use external knowledge, assumptions, or inferred facts. If the available evidence is insufficient, you must clearly explain why.

----------------------------------------------------------------
CORE PRINCIPLES
----------------------------------------------------------------
- Use only the provided text, images, and captions.
- Prioritize visual evidence from images if they directly relate to the text.
- Do not introduce outside knowledge.
- Do not assume missing details.
- If the text does not explicitly support the answer, clearly state that the context is insufficient.
- If an image or caption contradicts the text, clearly explain the inconsistency.
- If an image is unrelated to the object described in the text, explicitly state that it is not relevant.
- Before evaluating relevance, verify that the image depicts the SAME object or phenomenon mentioned in the text.
- Emphasize captions as they provide direct context to images.

----------------------------------------------------------------
REQUIRED STRUCTURE - USE CHAIN-OF-THOUGHT REASONING
----------------------------------------------------------------
1) Textual Evidence Assessment
   - Identify the specific object(s), phenomenon, or event described in the text.
   - Determine whether the text explicitly supports the question.
   - Summarize the exact supporting statements.
   - If the text does not adequately support the answer, explain why and stop.

2) Image and Caption Evaluation
   - Images provided: Yes / No
   - For each image: Identify what object or phenomenon is shown, describe the caption if present.
   - Compare it to the object described in the text.
   - State whether they refer to the same object.
   - If they refer to different objects, clearly state that the image is not relevant.
   - Describe only what is directly visible.
   - Conclude whether the image/caption:
     • Supports the text
     • Contradicts the text
     • Is unrelated or insufficient

3) Integrated Reasoning with Chain-of-Thought
   - Think step-by-step about how the evidence connects to the question.
   - Connect the validated textual evidence with any relevant visual/caption evidence.
   - Explain mechanisms, processes, and any numerical details mentioned.
   - Identify logical steps that link evidence to conclusion.
   - Show your reasoning process explicitly.
   - Explicitly mention any limitations or missing information.

4) Final Conclusion
   - Provide a well-structured, natural explanation based on your reasoning.
   - Minimum 8–12 detailed sentences.
   - The conclusion must strictly follow from validated evidence.
   - Do not introduce any information not present in the provided material.
   - Cite specific sources when making claims (e.g., "According to [Source], page [X]...").

----------------------------------------------------------------
Context: {context}

Question: {query}

Answer (think step-by-step):""".strip()

    # The following function is used to convert PIL image to base64 since LLM models can read base64 or need image path.
    @staticmethod
    def pil_to_base64(img: Image.Image) -> str:
        buffer = io.BytesIO()
        img = img.convert("RGB")
        img.save(buffer, format="PNG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    # The following function is used to generate response from language model.
    def generate_response(self, query: str, context: str, images: Optional[List[Dict]] = None,
                         stream: bool = True, temperature: float = 0.7, max_tokens: int = 500) -> Dict:
        if not query or not query.strip():  # Query validation.
            raise ValueError("Query cannot be empty")
        if not context or not context.strip():
            if not images:
                raise ValueError("Context cannot be empty when no images are provided")  # Context validation.
        if len(context) > 10000:  # Checking if context is too large.
            print("Warning: Large context may be slow")

        try:
            prompt = self.build_prompt(query, context)  # Building a prompt using query and context.
            image_payload = []
            if images:
                for img_dict in images:
                    img = img_dict.get("image")
                    if isinstance(img, Image.Image):
                        image_payload.append(self.pil_to_base64(img))
                    else:
                        raise TypeError("Images must be PIL.Image.Image")

            # Context length . M9 (Input context length)
            context_chars = len(context)
            context_tokens = len(self.tokenizer.encode(context))

            start_time = time.perf_counter()
            response = ollama.chat(  # Getting response from model.
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a grounded assistant that answers only from provided text and images. Think step-by-step and show your reasoning."},
                    {
                        "role": "user",
                        "content": prompt,
                        "images": image_payload if image_payload else None
                    }
                ],
                stream=stream,
                options={
                    'temperature': temperature,
                    "num_predict": max_tokens
                },
                keep_alive=0
            )

            if stream:  # Displaying output through streaming.
                print(f"\n{'='*80}")
                print(f"QUERY: {query}")
                print(f"{'='*80}")
                print("ANSWER:")
                print("-" * 80)
                full_response = ""
                try:
                    for chunk in response:  # Displaying response as model gives output.
                        content = chunk.get("message", {}).get("content", "")
                        if content:
                            print(content, end="", flush=True)
                            full_response += content
                    print("\n" + "=" * 80)
                except Exception as e:
                    print(f"\nError during streaming: {e}")
                    raise
                final_response = full_response
            else:
                final_response = response["message"]["content"]  # If stream is off, give output all at once.

            generation_time = time.perf_counter() - start_time

            # ──────────────────── M13: Factual Consistency Distance (FCD) ───────────────────────────────
            fcd = None
            try:
                resp_emb = self.text_embedder.embed_query(final_response)
                ctx_emb = self.text_embedder.embed_query(prompt)
                sim = util.cos_sim(resp_emb, ctx_emb).item()
                dist = 1 - sim
                fcd = dist * 100
            except Exception as e:
                print(f"FCD computation failed: {e}")
                fcd = None

            # ────────────── M14: Faithfulness / Citation Recall ───────────────────────────────
            faithfulness = 0.0
            try:
                response_lower = final_response.lower()
                cited_sources = set(re.findall(r'(source:\s*[^,\n]+?|\w+\.pdf|https?://[^\s<>\n]+)', response_lower, re.IGNORECASE))
                cited_pages = set(re.findall(r'page\s*(\d+)', response_lower, re.IGNORECASE))
                cited_images = set(re.findall(r'(image\s*\d+|caption|figure\s*\d+)', response_lower, re.IGNORECASE))

                context_lower = context.lower()
                retrieved_sources = set(re.findall(r'(source:\s*[^,\n]+?|\w+\.pdf|https?://[^\s<>\n]+)', context_lower, re.IGNORECASE))
                retrieved_pages = set(re.findall(r'page\s*(\d+)', context_lower, re.IGNORECASE))
                retrieved_images = set(re.findall(r'(image\s*\d+|caption|figure\s*\d+)', context_lower, re.IGNORECASE))

                total_retrieved = len(retrieved_sources | retrieved_pages | retrieved_images) or 1
                total_cited = len(cited_sources | cited_pages | cited_images)
                faithfulness = (total_cited / total_retrieved) * 100
            except Exception as e:
                print(f"M14 Faithfulness computation failed: {e}")
                faithfulness = 0.0

            return {
                "response": final_response,
                "context_length_chars": context_chars,
                "context_length_tokens": context_tokens,
                "generation_time_sec": round(generation_time, 4),
                "factual_consistency_distance": round(fcd, 2) if fcd is not None else None,
                "faithfulness_percentage": round(faithfulness, 2)
            }
        except Exception as e:
            raise RuntimeError(f"Error generating response: {e}") from e

In [8]:
llm=LocalLLM() # Initializing model .

Ollama server started


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE running on NVIDIA GeForce RTX 3050 6GB Laptop GPU.
Embedding dimension of BAAI/bge-large-en-v1.5 is 1024


In [9]:
query="Why does the visual geometry prove that gravity assists fundamentally redirect trajectories rather than merely accelerate the spacecraft, and what irreversible decision did this force upon Voyager 1 at Saturn?"

In [10]:
content="""1] Voyager’s Grand Tour\nBy John Uri \nManager, History Office \nNASA Johnson Space Center \nhttps://www.jsc.nasa.gov/history/\nIn the early days of the Space Age, scientists realized that given the right planetary alignments it might \nbe possible to use the gravity of one planet to change the trajectory of a spacecraft and send it on to \nanother planet without expending any fuel.  This slingshot or gravity assist trajectory principle was first \ntested by Mariner 10, which used the gravity of Venus to slingshot its way to Mercury in 1974.\nA very rare planetary alignment would occur in the late 1970’s allowing a spacecraft to visit all the outer \nplanets (Jupiter, Saturn, Uranus, Neptune and Pluto) using gravity assists at each planet to send it on to \nthe next.  This unique alignment would not occur again for another 175 years!  The initial ambitious \nplan, called the Grand Tour, was to send two pairs of spacecraft, one pair to visit Jupiter, Saturn and \nPluto, the other to fly by Jupiter, Uranus and Neptune.  However, the original plan was scaled back in \nthe budget conscious early 1970’s to just two less capable spacecraft visiting only Jupiter and Saturn, \nand Titan, Saturn’s largest moon.\n(Source: Voyager Grand Tour.pdf, page 1)\n\n[2] Flight trajectories of Voyager 1 and 2 through the outer solar system (left) and the launch on top of a \nTitan III-Centaur rocket (right).  Images courtesy of NASA.\nTaking advantage of this alignment would be two Voyager spacecraft, both beginning their long journeys \nin 1977.  Voyager 2 launched first, on August 20, followed by Voyager 1 on September 5.  Both \nspacecraft would first fly by Jupiter and use that planet’s massive gravity to bend their trajectories to \nthen fly by Saturn.  Voyager 1 would also be targeted to fly by Saturn’s moon Titan, which was known to \nhave a dense atmosphere, a trajectory that would preclude any future planetary flybys.  But the option\n(Source: Voyager Grand Tour.pdf, page 1)\n\n[3] clues in the rocks about possible past life.\nEngineers designed the spacecraft to steer it\xad\nMore than 400 scientists from around the\nself during descent through Mars’ atmosphere\nworld participate in the science operations.\nwith a series of S-curve maneuvers similar to\nthose used by astronauts piloting NASA space\nshuttles. During the three minutes before touch\xad\ndown, the spacecraft slowed its descent with\na parachute, then used retrorockets mounted\naround the rim of its upper stage. In the final\nseconds, the upper stage acted as a sky crane,\nlowering the upright rover on a tether to land on\nThe touchdown site, Bradbury Landing, is near\nthe foot of a layered mountain, Aeolis Mons\nThe rover’s landing site, Gale Crater, is about the size of \nConnecticut and Rhode Island combined.\n(“Mount Sharp”). Selection of Gale Crater fol\xad\n(Source: mars-science-laboratory.pdf, page 1)"""

In [11]:
from json import dumps

In [12]:
# Test data .
print(dumps(llm.generate_response(query=query,context=content),indent=2,sort_keys=False)) # Pretty print


QUERY: Why does the visual geometry prove that gravity assists fundamentally redirect trajectories rather than merely accelerate the spacecraft, and what irreversible decision did this force upon Voyager 1 at Saturn?
ANSWER:
--------------------------------------------------------------------------------
Okay, let's analyze the provided text and images to answer the question about gravity assists and Voyager 1’s decision at Saturn.

1) **Textual Evidence Assessment:**
   - The text describes the “Grand Tour” mission, aiming to visit all outer planets using gravity assists.
   - It explains that gravity assists bend trajectories, utilizing a spacecraft's momentum and the gravitational pull of a planet to alter its course without expending fuel.
   - It states that Voyager 2 launched first, followed by Voyager 1, and both spacecraft would fly by Jupiter and Saturn, with Voyager 1 targeting Titan.
   - The text mentions S-curve maneuvers mimicking NASA space shuttle piloting, used for st

In [13]:
llm.ollama_server(process="stop") # Stopping server .

Ollama server successfully stopped.
